# **Problema 1**

Dado un hash encontrar la secuencia de números que lo genera.
<br>Reglas:
* hash → sha 256.
* La secuencia de números son 10 enteros concatenados, cada uno en el rango [0,9].

In [13]:
import hashlib
import multiprocessing as mp
from tqdm.notebook import tqdm  # opcional, instalar con !pip install tqdm

def search_chunk(args):
    start, end, target_hash = args
    target = bytes.fromhex(target_hash)
    for i in range(start, end):
        s = f"{i:010d}"
        if hashlib.sha256(s.encode()).digest() == target:
            return s
    return None

def find_parallel(target_hash, num_processes=None):
    total = 10**10
    if num_processes is None:
        num_processes = mp.cpu_count()
    chunk_size = total // num_processes
    ranges = []
    for i in range(num_processes):
        start = i * chunk_size
        end = total if i == num_processes-1 else (i+1)*chunk_size
        ranges.append((start, end, target_hash))

    with mp.Pool(num_processes) as pool:
        # Usar imap_unordered para obtener resultados a medida que llegan
        for result in pool.imap_unordered(search_chunk, ranges):
            if result is not None:
                pool.terminate()
                return result
    return None

if __name__ == "__main__":
    target = input("Hash: ").strip()
    res = find_parallel(target)
    print("Secuencia:", res)

Hash: 07963a0301a24ee7906c98f1c965a7b6d58328a7f664ef89c92ce3001935ffac
Secuencia: 0001000000


# **Problema 2**

Dado el hash root del árbol de Merkle determinar el orden de las transacciones que lo generan.
<br>Reglas:

* Las transacciones son conocidas, y el número de transacciones.

In [33]:
import hashlib
import itertools
from typing import List, Optional

def merkle_root(leaf_hashes: List[bytes]) -> bytes:
    """
    Calcula la raíz de Merkle a partir de una lista de hashes de hojas (bytes).
    Si la lista está vacía, retorna b''.
    Para niveles impares, duplica el último elemento.
    """
    if not leaf_hashes:
        return b''
    level = leaf_hashes[:]  # copia
    while len(level) > 1:
        next_level = []
        for i in range(0, len(level), 2):
            left = level[i]
            right = level[i+1] if i+1 < len(level) else left
            combined = left + right
            next_level.append(hashlib.sha256(combined).digest())
        level = next_level
    return level[0]

def find_permutation(transactions: List[str], target_root_hex: str) -> Optional[List[str]]:
    """
    Busca una permutación de las transacciones cuyo Merkle root sea igual a target_root_hex.
    Retorna la lista de transacciones en el orden encontrado, o None si no existe.
    """
    # Precomputar hashes de cada transacción (bytes)
    leaf_hashes = [hashlib.sha256(tx.encode('utf-8')).digest() for tx in transactions]
    target = bytes.fromhex(target_root_hex)
    n = len(transactions)

    # Probar todas las permutaciones de los índices
    for perm_indices in itertools.permutations(range(n)):
        # Construir lista de hashes en el orden de la permutación
        ordered_hashes = [leaf_hashes[i] for i in perm_indices]
        root = merkle_root(ordered_hashes)
        if root == target:
            # Devolver las transacciones en ese orden
            return [transactions[i] for i in perm_indices]
    return None

# --- Ejemplo de uso (prueba) ---
if __name__ == "__main__":
    # Datos de ejemplo: 4 transacciones
    transacciones = ["A", "B", "C", "D"]

    # Elegimos un orden conocido para generar la raíz objetivo
    orden_conocido = ["B", "A", "D", "C"]
    hashes_orden = [hashlib.sha256(tx.encode()).digest() for tx in orden_conocido]
    root_objetivo = merkle_root(hashes_orden).hex()

    print("Transacciones:", transacciones)
    print("Raíz objetivo (hex):", root_objetivo)

    # Buscar el orden que produce esa raíz
    resultado = find_permutation(transacciones, root_objetivo)
    if resultado:
        print("Orden encontrado:", resultado)
    else:
        print("No se encontró ningún orden.")

Transacciones: ['A', 'B', 'C', 'D']
Raíz objetivo (hex): cea2061b7cde1d598b770fb76d0fb5a300a3b032dec5d86c21cb1731832ed2a8
Orden encontrado: ['B', 'A', 'D', 'C']
